In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score
)

In [3]:
df = pd.read_csv("../../data/processed/master_dataset.csv")

print(df.shape)

df.head()

(449505, 25)


,state_name,district_name,year,season,crop,area,production,yield,Boron,Copper,...,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDHRA PRADESH,CHITTOOR,1997,Kharif,Arhar/Tur,6100.0,900.0,0.147541,12.343402,12.2969,...,8.231055,8.231055,8.216216,12.338633,12.327107,24.919123,852.26,68.333205,2.833452,18.610904
1,ANDHRA PRADESH,CHITTOOR,1997,Kharif,Bajra,1700.0,1900.0,1.117647,12.343402,12.2969,...,8.231055,8.231055,8.216216,12.338633,12.327107,24.919123,852.26,68.333205,2.833452,18.610904
2,ANDHRA PRADESH,CHITTOOR,1997,Kharif,Dry chillies,600.0,800.0,1.333333,12.343402,12.2969,...,8.231055,8.231055,8.216216,12.338633,12.327107,24.919123,852.26,68.333205,2.833452,18.610904
3,ANDHRA PRADESH,CHITTOOR,1997,Kharif,Groundnut,234900.0,144200.0,0.613878,12.343402,12.2969,...,8.231055,8.231055,8.216216,12.338633,12.327107,24.919123,852.26,68.333205,2.833452,18.610904
4,ANDHRA PRADESH,CHITTOOR,1997,Kharif,Horse-gram,3100.0,2000.0,0.645161,12.343402,12.2969,...,8.231055,8.231055,8.216216,12.338633,12.327107,24.919123,852.26,68.333205,2.833452,18.610904


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 449505 entries, 0 to 449504
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   state_name               449505 non-null  str    
 1   district_name            449505 non-null  str    
 2   year                     449505 non-null  int64  
 3   season                   449505 non-null  str    
 4   crop                     449505 non-null  str    
 5   area                     449505 non-null  float64
 6   production               449505 non-null  float64
 7   yield                    449505 non-null  float64
 8   Boron                    449505 non-null  float64
 9   Copper                   449505 non-null  float64
 10  Electrical Conductivity  449505 non-null  float64
 11  Iron                     449505 non-null  float64
 12  Manganese                449505 non-null  float64
 13  Nitrogen                 449505 non-null  float64
 14  Organic Carbon 

In [5]:
df.columns.tolist()

['state_name',
 'district_name',
 'year',
 'season',
 'crop',
 'area',
 'production',
 'yield',
 'Boron',
 'Copper',
 'Electrical Conductivity',
 'Iron',
 'Manganese',
 'Nitrogen',
 'Organic Carbon',
 'Phosphorus',
 'Potassium',
 'Soil Ph',
 'Sulphur',
 'Zinc',
 'temperature',
 'rainfall',
 'humidity',
 'wind_speed',
 'solar_radiation']

In [6]:
X = df.drop(
    columns=[
        "crop",
        "area",
        "production",
        "yield"
    ]
)

y = df["crop"]

print("Feature shape:", X.shape)
print("Target shape :", y.shape)

Feature shape: (449505, 21)
Target shape : (449505,)


In [7]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['state_name', 'district_name', 'season']

Numerical Features:
['year', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']


In [11]:
import time

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [12]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [13]:
results = []

def evaluate_model(name, model):
    start = time.time()

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    train_time = time.time() - start

    start = time.time()

    y_pred = pipeline.predict(X_test)

    prediction_time = time.time() - start

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (s)": train_time,
        "Prediction Time (s)": prediction_time
    })

    print(f"{name} completed.")
    return pipeline

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(359604, 21)
(89901, 21)


In [17]:
print("X_train" in globals())
print("y_train" in globals())

True
True


In [18]:
dt_model = evaluate_model(
    "Decision Tree",
    DecisionTreeClassifier(random_state=42)
)

Decision Tree completed.


In [19]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s),Prediction Time (s)
0,Decision Tree,0.029944,0.08891,0.029944,0.041197,108.255916,0.186697


In [22]:
print(df["crop"].value_counts().describe())

count      126.000000
mean      3567.500000
std       5707.577742
min          4.000000
25%         36.000000
50%        270.000000
75%       5934.750000
max      27693.000000
Name: count, dtype: float64


In [23]:
thresholds = [100, 250, 500, 1000]

for t in thresholds:
    n_crops = (df["crop"].value_counts() >= t).sum()
    n_rows = df[df["crop"].map(df["crop"].value_counts()) >= t].shape[0]

    print(f"Threshold: {t}")
    print(f"  Crops retained : {n_crops}")
    print(f"  Rows retained  : {n_rows:,}")
    print("-" * 40)

Threshold: 100
  Crops retained : 74
  Rows retained  : 447,765
----------------------------------------
Threshold: 250
  Crops retained : 64
  Rows retained  : 445,960
----------------------------------------
Threshold: 500
  Crops retained : 56
  Rows retained  : 443,215
----------------------------------------
Threshold: 1000
  Crops retained : 54
  Rows retained  : 441,632
----------------------------------------


In [24]:
duplicates = df.duplicated(
    subset=[
        "state_name",
        "district_name",
        "year",
        "season",
        "Boron",
        "Copper",
        "Electrical Conductivity",
        "Iron",
        "Manganese",
        "Nitrogen",
        "Organic Carbon",
        "Phosphorus",
        "Potassium",
        "Soil Ph",
        "Sulphur",
        "Zinc",
        "temperature",
        "rainfall",
        "humidity",
        "wind_speed",
        "solar_radiation"
    ],
    keep=False
)

print("Duplicate feature rows:", duplicates.sum())

Duplicate feature rows: 447505


In [25]:
check = (
    df.groupby(["state_name", "district_name", "year", "season"])["crop"]
      .nunique()
      .sort_values(ascending=False)
)

print(check.head(20))
print("\nMaximum crops in one district-year-season:", check.max())

state_name  district_name  year  season    
TAMIL NADU  DINDIGUL       2002  Whole Year    49
                           2003  Whole Year    49
            DHARMAPURI     2002  Whole Year    46
            SALEM          2003  Whole Year    45
            ERODE          2003  Whole Year    43
            COIMBATORE     2003  Whole Year    42
            THENI          2003  Whole Year    41
            KRISHNAGIRI    2003  Whole Year    41
            ERODE          2002  Whole Year    41
            VELLORE        2003  Whole Year    40
            THENI          2002  Whole Year    40
            SALEM          2002  Whole Year    39
            VELLORE        2002  Whole Year    39
            COIMBATORE     2002  Whole Year    38
            MADURAI        2003  Whole Year    38
            THE NILGIRIS   2002  Whole Year    38
            DHARMAPURI     2003  Whole Year    36
            TIRUNELVELI    2002  Whole Year    36
                           2003  Whole Year    36
      

In [26]:
grouped = df.groupby([
    "state_name",
    "district_name",
    "year",
    "season"
])

print("Unique environment combinations:", grouped.ngroups)

Unique environment combinations: 39268


In [1]:
print(X.columns.tolist())

NameError: name 'X' is not defined